# 02 - Rubric 评估标准

**来源:** [LangChain Docs - Grading Rubrics](https://docs.langchain.com/oss/python/deepagents/rubric)

RubricMiddleware 让 Agent 能够**自我评估并迭代**，直到满足评分标准或到达最大迭代次数。

## 1. 工作流程

```mermaid
graph LR
    Start[User invokes with rubric] --> Agent[Deep agent]
    Agent --> Grader{Grader verdict}
    Grader --> |satisfied| Done[Finish execution]
    Grader --> |failed| Done
    Grader --> |grader_error| Done
    Grader --> |needs_revision| Cap{Iterations < max_iterations?}
    Cap --> |yes| Inject[Re-prompt with feedback]
    Cap --> |no| Done
    Inject --> Agent
```

Agent 完成推理和输出后，一个独立的 Grader 子 Agent 会对照 Rubric 审查。如果返回 `needs_revision`，反馈会被注入对话，Agent 重新运行。
循环在 `satisfied`、`max_iterations_reached`、`failed` 或 `grader_error` 时终止。

## 2. Rubric Verdict(评分结论)

| 结果 | 含义 | 是否循环回 Agent？ |
|------|------|:-----------------:|
| satisfied | 所有标准通过 | 否 |
| needs_revision | 至少一条标准未通过，注入反馈重新运行 | 是 |
| max_iterations_reached | 已达到最大迭代次数 | 否 |
| failed | Rubric 格式错误或无法评估 | 否 |
| grader_error | Grader 子 Agent 异常(超时/凭据/格式) | 否 |

## 3. 配置选项

| 参数 | 必须 | 默认值 | 说明 |
|------|:----:|--------|------|
| model | 是 | None | Grader 使用的 Chat Model(字符串或实例) |
| system_prompt | 否 | 内置 prompt | 自定义评分指令 |
| tools | 否 | None | Grader 可调用的工具(运行测试、数 token、读文件) |
| max_iterations | 否 | 3 | 硬上限，最大 20 |
| on_evaluation | 否 | None | 每次评分后的回调 |

## 4. 完整示例: 避诗体(Lipogram)评分

构建一个写海洋段落但不允许使用字母 'e' 的 Agent：

In [ ]:
from deepagents import RubricMiddleware, create_deep_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
load_dotenv()

@tool
def check_lipogram(text: str, forbidden: str = "e") -> dict:
    """检查文本是否包含禁用的字母 (case-insensitive)."""
    forbidden_lower = forbidden.lower()
    violating_words = sorted({w for w in text.split() if forbidden_lower in w.lower()})
    hit_count = sum(1 for c in text.lower() if c == forbidden_lower)
    return {
        "forbidden": forbidden,
        "hit_count": hit_count,
        "violating_words": violating_words,
        "ok": hit_count == 0,
    }

LIPOGRAM_GRADER_PROMPT = "你是一名编辑，负责对短文进行评分 against a rubric."

agent = create_deep_agent(
    model="deepseek-v4-flash",
    middleware=[
        RubricMiddleware(
            model="deepseek-v4-flash",
            system_prompt=LIPOGRAM_GRADER_PROMPT,
            tools=[check_lipogram],
            max_iterations=5,
            on_evaluation=lambda ev: print(ev["result"], ev["explanation"]),
        ),
    ],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "lipogram-session"}}
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                "Write a short paragraph (3-4 sentences) about the ocean without using the letter 'e'."
            )
        ],
        "rubric": (
            "- The paragraph contains zero instances of the letter 'e'\\n"
            "- The paragraph is about the ocean and includes at least two vivid sensory details\\n"
            "- The paragraph is 3-4 sentences long."
        ),
    },
    config=config,
)

print(result["messages"][-1].content)


## 5. 观察迭代进度

使用 `agent.stream(stream_mode="custom")` 可以实时接收每次评分事件：

| 事件 | 触发时机 | 关键字段 |
|------|----------|----------|
| rubric_evaluation_start | Grader 运行前 | grading_run_id, iteration |
| rubric_evaluation_end | Grader 返回后 | result, explanation, criteria |

每次评分迭代的 `RubricEvaluation` 也会同步回调到 `on_evaluation`。完整历史可通过 `agent.get_state(config).values` 在有 checkpoint 的线程上获取。

---
**参考链接**

- [Grading Rubrics - LangChain Docs](https://docs.langchain.com/oss/python/deepagents/rubric)
- [Streaming](https://docs.langchain.com/oss/python/deepagents/streaming)
- [Human-in-the-loop](https://docs.langchain.com/oss/python/deepagents/human-in-the-loop)